# OLMo 3 7B Think Goal 1 Colab Benchmark

Purpose: run a small `carry-trace` Goal 1 dataset through `allenai/Olmo-3-7B-Think` on a Colab A100 and use the measured throughput to estimate realistic paper-like runtime.

The benchmark configs are defined in this notebook. The repo is cloned or pulled only for the package code.

## 1. Clone Or Update Repo

Edit `REPO_URL` if the GitHub remote is different or private. If the repo is private, authenticate with Colab/GitHub before running this cell.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ndalton12/carry-trace.git"  # Change if needed.
BRANCH = "main"
REPO_DIR = Path("/content/carry-trace")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Repo: {REPO_DIR}")
print(f"Commit: {commit}")
print(f"Python: {sys.version}")

## 2. Install Dependencies

Colab usually already has CUDA PyTorch. This installs the repo and its Python dependencies. `--ignore-requires-python` is included so the notebook still works if Colab's Python is slightly behind the repo's declared Python version.

In [ ]:
!python -m pip install -U pip
!python -m pip install -e . --ignore-requires-python

## 3. GPU And Hugging Face Setup

If the model requires auth in your environment, set `HF_TOKEN` in Colab secrets or `os.environ` before running the login cell.

In [ ]:
import os
import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("bf16 supported", torch.cuda.is_bf16_supported())

!nvidia-smi

In [ ]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged into Hugging Face using HF_TOKEN.")
else:
    print("No HF_TOKEN set. Continuing unauthenticated.")

## 4. Define Benchmark Configs In Notebook

Adjust these variables to trade off runtime and estimate quality. The defaults intentionally create a small but mixed benchmark.

In [ ]:
from pathlib import Path
import json
import yaml

BENCH_NAME = "colab_olmo3_think_benchmark"
SEED = 20260604
EXAMPLES_PER_SLICE_PER_LENGTH = 1
MAX_NEW_TOKENS = 128

DIGIT_LENGTHS = [2, 4, 8]
SLICES = ["no_carry", "isolated_carry", "long_carry_chain"]
PROMPT_MODES = ["answer_only", "structured_column_cot"]
DIGIT_FORMATS = ["plain", "delimited"]
ANSWER_FORMATS = ["standard", "lsd_delimited"]

dataset_config = {
    "name": BENCH_NAME,
    "seed": SEED,
    "base": 10,
    "output_dir": "data/generated",
    "write_parquet": False,
    "schema_version": "goal1.v1",
    "splits": {
        "benchmark": {
            "examples_per_slice_per_length": EXAMPLES_PER_SLICE_PER_LENGTH,
        },
    },
    "digit_lengths": DIGIT_LENGTHS,
    "slices": SLICES,
    "prompt_modes": PROMPT_MODES,
    "digit_formats": DIGIT_FORMATS,
    "answer_formats": ANSWER_FORMATS,
    "digit_delimiter": "|",
    "answer_delimiter": "|",
}

experiment_config = {
    "name": BENCH_NAME,
    "seed": SEED,
    "dataset_path": f"data/generated/{BENCH_NAME}/examples.jsonl",
    "output_dir": "runs",
    "splits": ["benchmark"],
    "prompt_modes": PROMPT_MODES,
    "digit_formats": DIGIT_FORMATS,
    "answer_formats": ANSWER_FORMATS,
    "runner": {
        "kind": "hf",
        "device": "auto",
        "dtype": "bfloat16",
        "batch_size": 1,
        "trust_remote_code": False,
    },
    "models": [
        {
            "name": "olmo3-think",
            "model_id": "allenai/Olmo-3-7B-Think",
        },
    ],
    "generation": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": 0.0,
        "top_p": 1.0,
        "do_sample": False,
    },
}

config_dir = Path("configs/colab")
config_dir.mkdir(parents=True, exist_ok=True)
dataset_config_path = config_dir / "dataset_benchmark.yaml"
experiment_config_path = config_dir / "experiment_olmo3_think_benchmark.yaml"
dataset_config_path.write_text(yaml.safe_dump(dataset_config, sort_keys=False), encoding="utf-8")
experiment_config_path.write_text(yaml.safe_dump(experiment_config, sort_keys=False), encoding="utf-8")

expected_rows = (
    len(DIGIT_LENGTHS)
    * len(SLICES)
    * EXAMPLES_PER_SLICE_PER_LENGTH
    * len(PROMPT_MODES)
    * len(DIGIT_FORMATS)
    * len(ANSWER_FORMATS)
)
print(f"Expected benchmark examples: {expected_rows}")
print("\nDataset config:\n", dataset_config_path.read_text())
print("\nExperiment config:\n", experiment_config_path.read_text())

## 5. Generate Dataset

In [ ]:
!carry-trace dataset generate --config configs/colab/dataset_benchmark.yaml

In [ ]:
from collections import Counter
from carry_trace.io import read_jsonl

dataset_path = Path(experiment_config["dataset_path"])
examples = read_jsonl(dataset_path)
print("rows", len(examples))
print("n_digits", Counter(row["n_digits"] for row in examples))
print("slice_name", Counter(row["slice_name"] for row in examples))
print("prompt_mode", Counter(row["prompt_mode"] for row in examples))
print("digit_format", Counter(row["digit_format"] for row in examples))
print("answer_format", Counter(row["answer_format"] for row in examples))
print("\nFirst prompt:\n", examples[0]["prompt"])
print("\nFirst expected_output:", examples[0]["expected_output"])

## 6. Run OLMo 3 7B Think

This cell loads the model and runs all benchmark examples. The printed wall time includes model loading. Per-example `latency_seconds` in `calls.jsonl` measures tokenization + generation after the model is loaded.

In [ ]:
import time
import subprocess

start = time.perf_counter()
proc = subprocess.run(
    ["carry-trace", "run", "goal1", "--config", str(experiment_config_path)],
    text=True,
    capture_output=True,
)
RUN_WALL_SECONDS = time.perf_counter() - start
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f"carry-trace run failed with exit code {proc.returncode}")

run_dirs = sorted(Path("runs").glob(f"{BENCH_NAME}-*"), key=lambda path: path.stat().st_mtime)
RUN_DIR = run_dirs[-1]
print("RUN_DIR", RUN_DIR)
print("wall_seconds", RUN_WALL_SECONDS)

## 7. Summarize Throughput And Accuracy

In [ ]:
import pandas as pd

calls = read_jsonl(RUN_DIR / "calls.jsonl")
scored = read_jsonl(RUN_DIR / "scored_calls.jsonl")
summary = pd.read_csv(RUN_DIR / "metrics_summary.csv")

total_calls = len(calls)
total_output_tokens = sum(row["token_count_output"] for row in calls)
total_input_tokens = sum(row["token_count_input"] for row in calls)
sum_generation_latency = sum(row["latency_seconds"] for row in calls)

print("run_dir", RUN_DIR)
print("calls", total_calls)
print("wall_seconds_including_model_load", round(RUN_WALL_SECONDS, 2))
print("sum_generation_latency_seconds", round(sum_generation_latency, 2))
print("examples_per_second_after_load", round(total_calls / sum_generation_latency, 4))
print("output_tokens_per_second_after_load", round(total_output_tokens / sum_generation_latency, 2))
print("avg_input_tokens", round(total_input_tokens / total_calls, 2))
print("avg_output_tokens", round(total_output_tokens / total_calls, 2))
print("parsed_accuracy", sum(row["parsed_answer_correct"] for row in scored) / len(scored))
print("output_format_accuracy", sum(row["parsed_output_format_correct"] for row in scored) / len(scored))

display(summary)

## 8. Print All Model Outputs

This prints every generated output in the benchmark, with enough metadata to inspect failures.

In [ ]:
examples_by_id = {row["id"]: row for row in read_jsonl(RUN_DIR / "dataset.jsonl")}

for idx, row in enumerate(scored, start=1):
    example = examples_by_id[row["example_id"]]
    print("=" * 100)
    print(f"Example {idx}/{len(scored)}")
    print(
        "metadata:",
        {
            "n_digits": example["n_digits"],
            "slice_name": example["slice_name"],
            "prompt_mode": example["prompt_mode"],
            "digit_format": example["digit_format"],
            "answer_format": example["answer_format"],
            "answer": example["answer"],
            "expected_output": example["expected_output"],
        },
    )
    print("prompt:", example["prompt"])
    print("rendered_prompt:", row["rendered_prompt"])
    print("decoded_output:")
    print(row["decoded_output"])
    print("parsed_output:", row.get("parsed_output"))
    print("parsed_answer:", row.get("parsed_answer"))
    print("parsed_answer_correct:", row.get("parsed_answer_correct"))
    print("parsed_output_format_correct:", row.get("parsed_output_format_correct"))

## 9. Estimate Paper-Like Runtime

This projects runtime from the measured per-example generation latency. It also reads the current checked-in paper-like dataset config if present.

In [ ]:
from carry_trace.config import load_dataset_config

def rendered_rows_for_dataset_config(config):
    split_replicates = sum(
        split.examples_per_slice_per_length
        if split.examples_per_slice_per_length is not None
        else config.examples_per_slice_per_length
        for split in config.splits.values()
    )
    return (
        len(config.digit_lengths)
        * len(config.slices)
        * split_replicates
        * len(config.prompt_modes)
        * len(config.digit_formats)
        * len(config.answer_formats)
    )

seconds_per_example = sum_generation_latency / total_calls
print("measured_seconds_per_example_after_load", round(seconds_per_example, 4))

paper_config_path = Path("configs/datasets/goal1_paper_like.yaml")
if paper_config_path.exists():
    paper_config = load_dataset_config(paper_config_path)
    paper_rows = rendered_rows_for_dataset_config(paper_config)
    print("paper_like_rows_one_checkpoint", paper_rows)
    for n_checkpoints in [1, 2]:
        estimated_seconds = paper_rows * n_checkpoints * seconds_per_example
        print(
            f"estimated_after_load_for_{n_checkpoints}_checkpoint(s): "
            f"{estimated_seconds / 3600:.2f} hours"
        )

print("\nUse this estimate cautiously: longer CoT outputs and different max_new_tokens can dominate runtime.")